# 06 - IV vs Realized Vol

Is IV overpricing or underpricing future realized volatility?

If IV > RV → options are expensive (sell vol).
If IV < RV → options are cheap (buy vol).


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load data

In [ ]:
s = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
s["date_dt"] = pd.to_datetime(s.ts, unit="ms")
s = s.set_index("date_dt")

px = mkt("SELECT open_time, close FROM ohlcv WHERE symbol='ETHUSDT' AND timeframe='1d' ORDER BY open_time")
px["date_dt"] = pd.to_datetime(px.open_time, unit="ms")
px = px.set_index("date_dt")["close"]
print("summary rows:", len(s))
print("price rows  :", len(px))

## 2. Compute realized vol (30d)

In [ ]:
r = px.pct_change()
rv = r.rolling(30).std() * np.sqrt(365)
rv.tail(3)

## 3. IV vs RV

In [ ]:
if not s.empty and s.atm_iv.notna().any():
    fig, ax = plt.subplots()
    ax.plot(s.index, s.atm_iv * 100, "o-", color="crimson", label="ATM IV")
    ax.plot(rv.index, rv * 100, "-", color="steelblue", label="30d realized vol")
    ax.set_title("IV vs realized vol")
    ax.set_ylabel("vol %")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Vol premium

In [ ]:
if not s.empty and s.atm_iv.notna().any():
    aligned = pd.concat([s.atm_iv.rename("iv"), rv.rename("rv")], axis=1).dropna()
    aligned["premium"] = aligned.iv - aligned.rv
    print(f"mean premium : {aligned.premium.mean()*100:+.2f} pp")
    print(f"last premium : {aligned.premium.iloc[-1]*100:+.2f} pp")
    aligned.tail()